# Member 2 — Encoding and Scaling

**Project:** Hotel Booking Demand — Cancellation Risk  
**Owner:** IT24103767 (Member 2)

Purpose: Apply feature encoding and scaling transformations.


## Cell 1 — Load the training data

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

data_path = Path("../../data/processed/hotel_bookings_validated_phase3.csv")
split_path = Path("../../results/splits/group_holdout_indices.npz")

if not data_path.exists():
    data_path = Path("data/processed/hotel_bookings_validated_phase3.csv")
    split_path = Path("results/splits/group_holdout_indices.npz")

df = pd.read_csv(data_path)
splits = np.load(split_path)

excluded = [
    "is_canceled", "assigned_room_type", "booking_changes",
    "deposit_type", "agent", "company", "adr",
    "required_car_parking_spaces", "total_of_special_requests",
    "reservation_status", "reservation_status_date"
]

X = df.drop(columns=excluded)

X_train = X.iloc[splits["train_idx"]].copy()
X_test = X.iloc[splits["test_idx"]].copy()

print("Training:", X_train.shape)
print("Testing:", X_test.shape)


Training: (95334, 21)
Testing: (23876, 21)


## Cell 2 — Identify numerical and categorical features

In [2]:
categorical = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numerical = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

print("Categorical features:", len(categorical))
print(categorical)

print("\nNumerical features:", len(numerical))
print(numerical)


Categorical features: 8
['hotel', 'arrival_date_month', 'meal', 'country', 'market_segment', 'distribution_channel', 'reserved_room_type', 'customer_type']

Numerical features: 13
['lead_time', 'arrival_date_year', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'days_in_waiting_list']


## Cell 3 — Create the preprocessing pipeline

In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Numerical preprocessing
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine both transformations
preprocessor = ColumnTransformer([
    ("num", num_pipeline, numerical),
    ("cat", cat_pipeline, categorical)
])

print("Preprocessing pipeline created successfully.")


Preprocessing pipeline created successfully.


## Cell 4 — Test the pipeline

In [4]:

from sklearn.base import clone

sample = X_train.iloc[:1000].copy()

test_pipeline = clone(preprocessor)
transformed = test_pipeline.fit_transform(sample)

print("Original shape:", sample.shape)
print("Transformed shape:", transformed.shape)
print("Transformed feature count:",
      len(test_pipeline.get_feature_names_out()))

print("\nFirst 10 transformed features:")
print(test_pipeline.get_feature_names_out()[:10])

print("\nPipeline test completed.")


Original shape: (1000, 21)
Transformed shape: (1000, 73)
Transformed feature count: 73

First 10 transformed features:
['num__lead_time' 'num__arrival_date_year' 'num__arrival_date_week_number'
 'num__arrival_date_day_of_month' 'num__stays_in_weekend_nights'
 'num__stays_in_week_nights' 'num__adults' 'num__children' 'num__babies'
 'num__is_repeated_guest']

Pipeline test completed.


## Test the reusable pipeline

In [5]:

import sys
from sklearn.base import clone

# Import shared preprocessing function
repo_root = data_path.resolve().parents[2]
sys.path.insert(0, str(repo_root))

from src.preprocessing.pipeline import build_preprocessor

shared_pipeline = build_preprocessor(X_train)

# Test on training data only
sample = X_train.iloc[:1000]
transformed = clone(shared_pipeline).fit_transform(sample)

print("Original shape:", sample.shape)
print("Transformed shape:", transformed.shape)
print("Shared pipeline test passed.")


Original shape: (1000, 21)
Transformed shape: (1000, 73)
Shared pipeline test passed.


## Encoding and Scaling Summary

The selected dataset contains 21 input features: 13 numerical and 8 categorical.

Numerical variables are processed using median imputation and StandardScaler. Categorical variables are encoded using One-Hot Encoding.

These transformations are combined using a ColumnTransformer to maintain a consistent preprocessing approach across the classification models.

The pipeline was tested using training observations only. During model development, preprocessing will be fitted within each training fold to prevent data leakage.